# Churn Prediction — Banorte XB

**Approach:** Merchant × calendar-month panel — one row per merchant per month.

| Step | Detail |
|---|---|
| Unit of analysis | `(merchant, year_month)` — genuinely independent observations |
| Data | All available months (not just the last 6) |
| Features | Lags (T-1, T-2), month-over-month momentum, 3-month rolling stats |
| Target | Volume AND txs both drop > 50 % / 40 % vs own 3-month trailing median |
| Model | Ridge Logistic Regression (`C=0.1`) — appropriate for small N |
| Validation | Walk-forward only — never k-fold on a time series |

## 1. Setup & Data Loading

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.4f}".format)

DATA_DIR = Path("../data/raw/banorte_xb")

In [2]:
N_MONTHS = None  # set to an integer to load only the N most recent months

all_files = sorted(DATA_DIR.rglob("*.parquet"))
print(f"Total parquet files available: {len(all_files)}")

files_to_load = all_files[-N_MONTHS:] if N_MONTHS else all_files
print(f"Loading: {len(files_to_load)} months")
for f in files_to_load:
    print(f"  {f.relative_to(DATA_DIR)}")

df = pd.concat([pd.read_parquet(f) for f in files_to_load], ignore_index=True)
print(f"\nRaw shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Date range: {df['CONFIRM_DATE'].min()} -> {df['CONFIRM_DATE'].max()}")

Total parquet files available: 28
Loading: 28 months
  2023\12.parquet
  2024\01.parquet
  2024\02.parquet
  2024\03.parquet
  2024\04.parquet
  2024\05.parquet
  2024\06.parquet
  2024\07.parquet
  2024\08.parquet
  2024\09.parquet
  2024\10.parquet
  2024\11.parquet
  2024\12.parquet
  2025\01.parquet
  2025\02.parquet
  2025\03.parquet
  2025\04.parquet
  2025\05.parquet
  2025\06.parquet
  2025\07.parquet
  2025\08.parquet
  2025\09.parquet
  2025\10.parquet
  2025\11.parquet
  2025\12.parquet
  2026\01.parquet
  2026\02.parquet
  2026\03.parquet

Raw shape: 34,591,378 rows x 19 columns
Date range: 2023-12-01 -> 2026-03-02


## 2. Preprocessing

Self-contained — same pipeline as notebook 03.
**Includes an incomplete-month filter**: any month where the last observed transaction date is not the last calendar day of that month is dropped before building the panel (avoids spurious churn labels from cut-off data).

In [3]:
# Drop unnecessary columns
df = df.drop(columns=["PROVIDER", "PAYMENT_AMOUNT"])

# Parse dates
df["CONFIRM_DATE"] = pd.to_datetime(df["CONFIRM_DATE"])
df["FEE_DATE"]     = pd.to_datetime(df["FEE_DATE"])

# Numeric conversion
numeric_cols = [
    "AMOUNT_GROSS", "NET_AMOUNT", "FEE_EBANX", "IVA_EBANX",
    "CUSTO_TRANSACAO", "AMOUNT_INSTALLMENT", "AMOUNT_IVA_INSTALLMENT",
    "INSTALLMENTS",
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Fill categorical nulls
df["CARD_ISSUER"] = df["CARD_ISSUER"].fillna("OTHER")
df["CARD_TYPE"]   = df["CARD_TYPE"].fillna("OTHER")

# Cascading CUSTO_TRANSACAO imputation
group_levels = [
    ["MERCHANT_NAME", "FEE_DATE", "CARD_TYPE", "CARD_ISSUER"],
    ["MERCHANT_NAME", "CARD_TYPE", "CARD_ISSUER"],
    ["MERCHANT_NAME", "CARD_TYPE"],
    ["MERCHANT_NAME"],
]
for level_cols in group_levels:
    still_null = df["CUSTO_TRANSACAO"].isna()
    if not still_null.any():
        break
    medians = df.groupby(level_cols)["CUSTO_TRANSACAO"].transform("median")
    df.loc[still_null, "CUSTO_TRANSACAO"] = medians[still_null]
if df["CUSTO_TRANSACAO"].isna().any():
    df["CUSTO_TRANSACAO"] = df["CUSTO_TRANSACAO"].fillna(df["CUSTO_TRANSACAO"].median())

# Rename to lowercase / no branding
df = df.rename(columns={
    "MERCHANT_NAME":          "merchant",
    "CONFIRM_DATE":           "confirm_date",
    "FEE_DATE":               "fee_date",
    "AMOUNT_GROSS":           "amt_gross",
    "NET_AMOUNT":             "amt_net",
    "FEE_EBANX":              "fee",
    "IVA_EBANX":              "iva",
    "CUSTO_TRANSACAO":        "fee_pct",
    "AMOUNT_INSTALLMENT":     "amt_installment",
    "AMOUNT_IVA_INSTALLMENT": "amt_iva_installment",
    "INSTALLMENTS":           "installments",
    "CARD_TYPE":              "card_type",
    "CARD_ISSUER":            "card_issuer",
    "CARD_SCHEME":            "card_scheme",
    "OPERATION_TYPE":         "op_type",
    "CAMARA_COMPENSACION":    "clearing_house",
    "AFILIACION":             "affiliation",
})

df["year_month"] = df["confirm_date"].dt.to_period("M")

print(f"Preprocessed: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Nulls remaining: {df.isna().sum().sum()}")
months = sorted(df["year_month"].unique())
print(f"Months ({len(months)}): {[str(m) for m in months]}")

Preprocessed: 34,591,378 rows x 18 columns
Nulls remaining: 0
Months (28): ['2023-12', '2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12', '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03']


In [14]:
# Drop incomplete months — allow up to 3 days tolerance for pipeline lag
# (e.g. Feb 2026 max date = Feb 27 is still considered complete)
COMPLETENESS_TOLERANCE_DAYS = 3

month_completeness = (
    df.groupby("year_month")["confirm_date"]
    .max()
    .rename("max_date")
    .reset_index()
)
month_completeness["days_in_month"] = month_completeness["max_date"].dt.days_in_month
month_completeness["days_short"]    = (
    month_completeness["days_in_month"] - month_completeness["max_date"].dt.day
)
month_completeness["is_complete"] = (
    month_completeness["days_short"] <= COMPLETENESS_TOLERANCE_DAYS
)

print("Month completeness check (tolerance = 3 days):")
print(
    month_completeness[["year_month", "max_date", "days_short", "is_complete"]]
    .to_string(index=False)
)

complete_months = month_completeness.loc[month_completeness["is_complete"], "year_month"]
n_dropped = (~month_completeness["is_complete"]).sum()
df = df[df["year_month"].isin(complete_months)].copy()

print(f"\nKept {complete_months.nunique()} complete months, dropped {n_dropped} incomplete.")

Month completeness check (tolerance = 3 days):
year_month   max_date  days_short  is_complete
   2023-12 2023-12-29           2         True
   2024-01 2024-01-31           0         True
   2024-02 2024-02-29           0         True
   2024-03 2024-03-31           0         True
   2024-04 2024-04-30           0         True
   2024-05 2024-05-31           0         True
   2024-06 2024-06-30           0         True
   2024-07 2024-07-31           0         True
   2024-08 2024-08-31           0         True
   2024-09 2024-09-30           0         True
   2024-10 2024-10-31           0         True
   2024-11 2024-11-30           0         True
   2024-12 2024-12-31           0         True
   2025-01 2025-01-31           0         True
   2025-02 2025-02-28           0         True
   2025-03 2025-03-31           0         True
   2025-04 2025-04-30           0         True
   2025-05 2025-05-31           0         True
   2025-06 2025-06-30           0         True
   2025-07 20

## 3. Monthly Panel Aggregation

One row per `(merchant, year_month)`. The `refund_ratio` is computed from `op_type` — inspect actual values first.

In [15]:
print("op_type value counts:")
print(df["op_type"].value_counts())

op_type value counts:
op_type
payment       32668065
refund         1823471
chargeback       99841
Name: count, dtype: int64


In [16]:
PANEL_COLS = ["merchant", "year_month"]

g      = df.groupby(PANEL_COLS)
df_inst = df[df["installments"] > 1]
g_inst  = df_inst.groupby(PANEL_COLS)

# Base aggregations
df_panel = g.agg(
    total_txs                =("amt_gross", "size"),
    avg_ticket               =("amt_gross", "mean"),
    median_ticket            =("amt_gross", "median"),
    total_amt_gross          =("amt_gross", "sum"),
    total_amt_net            =("amt_net",   "sum"),
    total_fee                =("fee",       "sum"),
    total_iva                =("iva",       "sum"),
    total_amt_installment    =("amt_installment",     "sum"),
    total_amt_iva_installment=("amt_iva_installment", "sum"),
)

# Installment features
installment_txs = g_inst.size().rename("installment_txs")
df_panel = df_panel.join(installment_txs, how="left")
df_panel["installment_txs"]  = df_panel["installment_txs"].fillna(0).astype(int)
df_panel["installment_ratio"] = df_panel["installment_txs"] / df_panel["total_txs"]

# Weighted average percentages
df_panel["avg_fee_pct"] = df_panel["total_fee"] / df_panel["total_amt_gross"]
df_panel["avg_iva_pct"] = np.where(
    df_panel["total_fee"] > 0, df_panel["total_iva"] / df_panel["total_fee"], 0
)

inst_gross = g_inst["amt_gross"].sum().rename("_inst_gross")
inst_amt   = g_inst["amt_installment"].sum().rename("_inst_amt")
inst_iva   = g_inst["amt_iva_installment"].sum().rename("_inst_iva")
df_panel = df_panel.join(inst_gross).join(inst_amt).join(inst_iva)
df_panel["avg_installment_pct"]     = np.where(df_panel["_inst_gross"] > 0, df_panel["_inst_amt"] / df_panel["_inst_gross"], 0)
df_panel["avg_iva_installment_pct"] = np.where(df_panel["_inst_amt"]   > 0, df_panel["_inst_iva"] / df_panel["_inst_amt"],   0)
df_panel = df_panel.drop(columns=["_inst_gross", "_inst_amt", "_inst_iva"])

# Refund ratio — adjust REFUND_KEYWORDS based on op_type values above
REFUND_KEYWORDS = "REFUND|REVERSAL|CHARGEBACK|CANCEL"
refund_mask = df["op_type"].str.upper().str.contains(REFUND_KEYWORDS, na=False)
refund_txs  = df[refund_mask].groupby(PANEL_COLS).size().rename("refund_txs")
df_panel = df_panel.join(refund_txs, how="left")
df_panel["refund_txs"]   = df_panel["refund_txs"].fillna(0).astype(int)
df_panel["refund_ratio"] = df_panel["refund_txs"] / df_panel["total_txs"]

df_panel = df_panel.reset_index().sort_values(PANEL_COLS).reset_index(drop=True)

print(f"Panel: {df_panel.shape[0]:,} merchant-months x {df_panel.shape[1]} columns")
print(f"Merchants: {df_panel['merchant'].nunique()},  Months: {df_panel['year_month'].nunique()}")

Panel: 433 merchant-months x 19 columns
Merchants: 29,  Months: 27


In [17]:
# ── Merchant coverage overview ─────────────────────────────────────────────
merchant_summary = df_panel.groupby("merchant").agg(
    first_month=("year_month", "min"),
    last_month =("year_month", "max"),
    n_months   =("year_month", "count"),
    avg_txs    =("total_txs",  "mean"),
    avg_vol    =("total_amt_gross", "mean"),
).sort_values("avg_vol", ascending=False)

latest_month    = df_panel["year_month"].max()
active_merchants = set(df_panel[df_panel["year_month"] == latest_month]["merchant"])

print(f"Total merchants: {len(merchant_summary)}  (active in {latest_month}: {len(active_merchants)})\n")
print(merchant_summary.round(0).to_string())

inactive = merchant_summary[~merchant_summary.index.isin(active_merchants)]
if len(inactive):
    print(f"\nMerchants absent from {latest_month} — likely churned in historical data:")
    print(inactive[["first_month", "last_month", "n_months", "avg_vol"]].to_string())

# ── Filter test / demo merchants ───────────────────────────────────────────
EXCLUDE_MERCHANTS = ["Demonstration Merchant"]
df_panel = df_panel[~df_panel["merchant"].isin(EXCLUDE_MERCHANTS)].copy()
print(f"\nExcluded {EXCLUDE_MERCHANTS}.")
print(f"Panel after filter: {df_panel.shape[0]:,} rows, {df_panel['merchant'].nunique()} merchants")

Total merchants: 29  (active in 2026-02: 18)

                           first_month last_month  n_months      avg_txs          avg_vol
merchant                                                                                 
Temu.com                       2024-01    2026-02        26 744,095.0000 324,236,598.0000
Amazon Retail MX               2023-12    2026-02        27 206,521.0000 255,859,195.0000
SHEIN Latam                    2024-02    2026-02        25 130,256.0000 138,617,149.0000
Alibaba ICBU                   2025-07    2026-02         8   6,742.0000  83,596,602.0000
SHEIN (FC)                     2024-01    2024-08         8  53,630.0000  60,592,265.0000
Google Play                    2025-08    2026-02         5 487,338.0000  49,438,700.0000
Shopee MX                      2024-01    2024-04         4  73,839.0000  24,731,636.0000
Adobe - Braintree              2024-01    2026-02        26  27,954.0000  13,571,664.0000
Canva                          2024-08    2026-02     

## 4. Feature Engineering

All features for row `(merchant, T)` use data from **T-1 and earlier only** — no leakage.

| Group | Features | Window |
|---|---|---|
| Lag | `lag1_*`, `lag2_*` for volume, txs, ticket, installment_ratio, refund_ratio | T-1, T-2 |
| Momentum | `mom_volume_1m`, `mom_txs_1m`, `mom_ticket_1m` | (T-1 - T-2) / T-2 |
| Rolling | `roll3_*_mean`, `roll3_*_cv` for volume and txs | mean/std over [T-3, T-2, T-1] |
| Relative | `vol_vs_roll3`, `txs_vs_roll3` | lag1 / roll3_mean |

In [18]:
df_panel = df_panel.sort_values(["merchant", "year_month"]).reset_index(drop=True)

# ── Lag features ──────────────────────────────────────────────────────────
for col in ["total_amt_gross", "total_txs", "avg_ticket", "installment_ratio", "refund_ratio"]:
    df_panel[f"lag1_{col}"] = df_panel.groupby("merchant")[col].shift(1)
    df_panel[f"lag2_{col}"] = df_panel.groupby("merchant")[col].shift(2)

# ── Month-over-month momentum ─────────────────────────────────────────────
df_panel["mom_volume_1m"] = (
    (df_panel["lag1_total_amt_gross"] - df_panel["lag2_total_amt_gross"])
    / df_panel["lag2_total_amt_gross"]
)
df_panel["mom_txs_1m"] = (
    (df_panel["lag1_total_txs"] - df_panel["lag2_total_txs"])
    / df_panel["lag2_total_txs"]
)
df_panel["mom_ticket_1m"] = (
    (df_panel["lag1_avg_ticket"] - df_panel["lag2_avg_ticket"])
    / df_panel["lag2_avg_ticket"]
)

# ── 3-month rolling stats over [T-3, T-2, T-1] ───────────────────────────
# x.shift(1).rolling(3) at position T => mean/std of x[T-1], x[T-2], x[T-3]
for col, label in [("total_amt_gross", "vol"), ("total_txs", "txs")]:
    df_panel[f"roll3_{label}_mean"] = df_panel.groupby("merchant")[col].transform(
        lambda x: x.shift(1).rolling(3).mean()
    )
    df_panel[f"roll3_{label}_std"] = df_panel.groupby("merchant")[col].transform(
        lambda x: x.shift(1).rolling(3).std()
    )
    df_panel[f"roll3_{label}_cv"] = (
        df_panel[f"roll3_{label}_std"] / df_panel[f"roll3_{label}_mean"]
    )

# Relative position: is the most recent month above/below the recent baseline?
df_panel["vol_vs_roll3"] = df_panel["lag1_total_amt_gross"] / df_panel["roll3_vol_mean"]
df_panel["txs_vs_roll3"] = df_panel["lag1_total_txs"]       / df_panel["roll3_txs_mean"]

print(f"Feature engineering done — {df_panel.shape[1]} total columns")

Feature engineering done — 40 total columns


## 5. Churn Target

```
churn[T] = 1  if  total_amt_gross[T] < 0.50 * median(vol[T-3:T-1])
                  AND total_txs[T]   < 0.60 * median(txs[T-3:T-1])
```

Both conditions must hold to avoid flagging one-off outlier months.
The reference (trailing 3-month median) is computed from T-1 and earlier — no leakage.

In [19]:
# Reference medians — same shift(1).rolling(3) pattern, no leakage
df_panel["ref_vol_median"] = df_panel.groupby("merchant")["total_amt_gross"].transform(
    lambda x: x.shift(1).rolling(3).median()
)
df_panel["ref_txs_median"] = df_panel.groupby("merchant")["total_txs"].transform(
    lambda x: x.shift(1).rolling(3).median()
)

# Dual-threshold churn label
df_panel["churn"] = (
    (df_panel["total_amt_gross"] < 0.50 * df_panel["ref_vol_median"]) &
    (df_panel["total_txs"]       < 0.60 * df_panel["ref_txs_median"])
).astype(int)

print("Churn label distribution:")
print(df_panel["churn"].value_counts())
print(f"\nOverall churn rate: {df_panel['churn'].mean():.2%}")
print("\nChurn events by month:")
print(df_panel.groupby("year_month")["churn"].agg(["sum", "mean"]).rename(columns={"sum": "n_churn", "mean": "rate"}))

Churn label distribution:
churn
0    373
1     40
Name: count, dtype: int64

Overall churn rate: 9.69%

Churn events by month:
            n_churn   rate
year_month                
2023-12           0 0.0000
2024-01           0 0.0000
2024-02           0 0.0000
2024-03           0 0.0000
2024-04           4 0.2353
2024-05           5 0.3333
2024-06           4 0.2500
2024-07           2 0.1429
2024-08           2 0.1333
2024-09           2 0.1429
2024-10           0 0.0000
2024-11           0 0.0000
2024-12           3 0.2000
2025-01           1 0.0714
2025-02           1 0.0714
2025-03           0 0.0000
2025-04           0 0.0000
2025-05           3 0.2000
2025-06           3 0.1875
2025-07           1 0.0588
2025-08           0 0.0000
2025-09           0 0.0000
2025-10           1 0.0556
2025-11           2 0.1053
2025-12           1 0.0526
2026-01           3 0.1579
2026-02           2 0.1176


## 6. Modeling Dataset

Drop rows where lag/rolling features are NaN (first ~3 months per merchant have no history).
Build a time index for the walk-forward loop.

In [20]:
FEATURE_COLS = [
    "lag1_total_amt_gross", "lag2_total_amt_gross",
    "lag1_total_txs",       "lag2_total_txs",
    "mom_volume_1m",        "mom_txs_1m",        "mom_ticket_1m",
    "roll3_vol_mean",       "roll3_vol_cv",
    "roll3_txs_mean",       "roll3_txs_cv",
    "vol_vs_roll3",         "txs_vs_roll3",
    "lag1_installment_ratio",
    "lag1_refund_ratio",
]
TARGET_COL = "churn"

# Drop rows that have no feature history yet
df_model = df_panel.dropna(subset=FEATURE_COLS + ["ref_vol_median", "ref_txs_median"]).copy()

# Ordinal time index for walk-forward slicing
usable_months  = sorted(df_model["year_month"].unique())
month_to_idx   = {m: i for i, m in enumerate(usable_months)}
df_model["t_idx"] = df_model["year_month"].map(month_to_idx)

print(f"Modeling dataset: {df_model.shape[0]:,} rows x {len(FEATURE_COLS)} features")
print(f"Usable months: {len(usable_months)}  ({usable_months[0]} -> {usable_months[-1]})")
print(f"Churn events:  {df_model[TARGET_COL].sum()} / {len(df_model)} ({df_model[TARGET_COL].mean():.2%})")

Modeling dataset: 331 rows x 15 features
Usable months: 24  (2024-03 -> 2026-02)
Churn events:  40 / 331 (12.08%)


## 7. Walk-Forward Validation

Train on months 0..T-1, validate on T. Minimum 3 training months.

**Models compared:** LogReg (Ridge), RandomForest, GradBoost, XGBoost (if installed), CatBoost (if installed)

**Metrics per fold:** AUC-ROC, AUC-PR (better for imbalanced data), Precision, Recall, F1, Brier Score

In [21]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    average_precision_score, precision_score, recall_score,
    f1_score, brier_score_loss,
)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost not installed — skipping")

try:
    from catboost import CatBoostClassifier
    HAS_CAT = True
except ImportError:
    HAS_CAT = False
    print("catboost not installed — skipping")

# ── Model registry ─────────────────────────────────────────────────────────
MODELS_REGISTRY = {
    "LogReg":       LogisticRegression(C=0.1, max_iter=1000, random_state=42, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=3, random_state=42, class_weight="balanced"),
    "GradBoost":    GradientBoostingClassifier(n_estimators=100, max_depth=2, learning_rate=0.05, subsample=0.8, random_state=42),
}
if HAS_XGB:
    MODELS_REGISTRY["XGBoost"] = XGBClassifier(
        n_estimators=100, max_depth=2, learning_rate=0.05, subsample=0.8,
        random_state=42, eval_metric="logloss", verbosity=0,
    )
if HAS_CAT:
    MODELS_REGISTRY["CatBoost"] = CatBoostClassifier(
        iterations=100, depth=2, learning_rate=0.05,
        random_state=42, verbose=0, auto_class_weights="Balanced",
    )

print(f"Models: {list(MODELS_REGISTRY.keys())}\n")

# ── Walk-forward loop ──────────────────────────────────────────────────────
MIN_TRAIN_MONTHS = 3
n_months         = len(usable_months)
all_results      = []
# pooled predictions across all folds per model (for threshold analysis)
pooled = {name: {"y_true": [], "y_proba": []} for name in MODELS_REGISTRY}

for val_t in range(MIN_TRAIN_MONTHS, n_months):
    train = df_model[df_model["t_idx"] <  val_t]
    val   = df_model[df_model["t_idx"] == val_t]

    X_train = train[FEATURE_COLS].values
    y_train = train[TARGET_COL].values
    X_val   = val[FEATURE_COLS].values
    y_val   = val[TARGET_COL].values

    scaler    = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s   = scaler.transform(X_val)

    has_signal = (y_val.sum() > 0) and (len(np.unique(y_val)) > 1)

    for name, base_clf in MODELS_REGISTRY.items():
        clf = clone(base_clf)
        if HAS_XGB and name == "XGBoost" and y_train.sum() > 0:
            clf.set_params(scale_pos_weight=(y_train == 0).sum() / y_train.sum())

        clf.fit(X_train_s, y_train)
        proba = clf.predict_proba(X_val_s)[:, 1]
        pred  = (proba >= 0.5).astype(int)

        pooled[name]["y_true"].extend(y_val.tolist())
        pooled[name]["y_proba"].extend(proba.tolist())

        all_results.append({
            "model":       name,
            "val_month":   str(usable_months[val_t]),
            "n_train":     len(y_train),
            "churn_train": int(y_train.sum()),
            "n_val":       len(y_val),
            "churn_val":   int(y_val.sum()),
            "auc_roc":     round(roc_auc_score(y_val, proba), 4)          if has_signal else np.nan,
            "auc_pr":      round(average_precision_score(y_val, proba), 4) if has_signal else np.nan,
            "precision":   round(precision_score(y_val, pred, zero_division=0), 4),
            "recall":      round(recall_score(y_val, pred, zero_division=0), 4),
            "f1":          round(f1_score(y_val, pred, zero_division=0), 4),
            "brier":       round(brier_score_loss(y_val, proba), 4),
            "flagged":     int(pred.sum()),
        })

results_df = pd.DataFrame(all_results)
print(f"Walk-forward done: {len(results_df)} fold-model evaluations  "
      f"({len(results_df) // len(MODELS_REGISTRY)} folds × {len(MODELS_REGISTRY)} models)")

Models: ['LogReg', 'RandomForest', 'GradBoost', 'XGBoost', 'CatBoost']

Walk-forward done: 105 fold-model evaluations  (21 folds × 5 models)


In [22]:
# ── Summary table: mean ± std over folds with churn events ────────────────
folds_with_churn = results_df[results_df["churn_val"] > 0]

summary = (
    folds_with_churn
    .groupby("model")[["auc_roc", "auc_pr", "precision", "recall", "f1", "brier"]]
    .agg(["mean", "std"])
    .round(4)
)
summary.columns = [f"{m}_{s}" for m, s in summary.columns]
summary = summary.sort_values("auc_roc_mean", ascending=False)

print("=== Model Comparison (mean ± std, folds with churn events) ===\n")
for mn in summary.index:
    r = summary.loc[mn]
    print(
        f"{mn:15s}"
        f"  AUC-ROC {r.auc_roc_mean:.3f}±{r.auc_roc_std:.3f}"
        f"  AUC-PR {r.auc_pr_mean:.3f}±{r.auc_pr_std:.3f}"
        f"  Prec {r.precision_mean:.3f}"
        f"  Rec {r.recall_mean:.3f}"
        f"  F1 {r.f1_mean:.3f}"
        f"  Brier {r.brier_mean:.4f}"
    )

# ── Threshold sensitivity (pooled predictions across ALL folds) ────────────
THRESHOLDS = [0.2, 0.3, 0.4, 0.5, 0.6]
thresh_rows = []
for name in MODELS_REGISTRY:
    y_true  = np.array(pooled[name]["y_true"])
    y_proba = np.array(pooled[name]["y_proba"])
    for t in THRESHOLDS:
        pred = (y_proba >= t).astype(int)
        thresh_rows.append({
            "model":     name,
            "threshold": t,
            "precision": round(precision_score(y_true, pred, zero_division=0), 3),
            "recall":    round(recall_score(y_true, pred, zero_division=0), 3),
            "f1":        round(f1_score(y_true, pred, zero_division=0), 3),
            "n_flagged": int(pred.sum()),
            "flag_rate": f"{pred.mean():.1%}",
        })

thresh_df = pd.DataFrame(thresh_rows)
print("\n=== Threshold Sensitivity (pooled across all validation folds) ===\n")
print(thresh_df.to_string(index=False))

=== Model Comparison (mean ± std, folds with churn events) ===

GradBoost        AUC-ROC 0.854±0.141  AUC-PR 0.727±0.245  Prec 0.440  Rec 0.494  F1 0.428  Brier 0.1082
RandomForest     AUC-ROC 0.844±0.160  AUC-PR 0.688±0.250  Prec 0.393  Rec 0.561  F1 0.440  Brier 0.1204
CatBoost         AUC-ROC 0.835±0.168  AUC-PR 0.696±0.268  Prec 0.316  Rec 0.583  F1 0.367  Brier 0.1479
XGBoost          AUC-ROC 0.830±0.161  AUC-PR 0.672±0.259  Prec 0.399  Rec 0.650  F1 0.439  Brier 0.1318
LogReg           AUC-ROC 0.752±0.177  AUC-PR 0.549±0.258  Prec 0.354  Rec 0.594  F1 0.400  Brier 0.1656

=== Threshold Sensitivity (pooled across all validation folds) ===

       model  threshold  precision  recall     f1  n_flagged flag_rate
      LogReg     0.2000     0.1000  0.9030 0.1800        280     90.3%
      LogReg     0.3000     0.1400  0.7740 0.2360        172     55.5%
      LogReg     0.4000     0.2280  0.6770 0.3410         92     29.7%
      LogReg     0.5000     0.3270  0.5160 0.4000         49   

In [23]:
# ── Permutation test ───────────────────────────────────────────────────────
# Globally shuffle churn labels N times, re-run walk-forward with LogReg.
# If the real AUC sits far above the permuted distribution → genuine signal.

N_PERMUTATIONS = 100
rng = np.random.default_rng(42)
perm_mean_aucs = []

for _ in range(N_PERMUTATIONS):
    perm_labels = rng.permutation(df_model[TARGET_COL].values)
    df_perm     = df_model.assign(**{TARGET_COL: perm_labels})
    fold_aucs   = []

    for val_t in range(MIN_TRAIN_MONTHS, n_months):
        train_p = df_perm[df_perm["t_idx"] <  val_t]
        val_p   = df_perm[df_perm["t_idx"] == val_t]
        # skip fold if train or val has only one class (can happen after shuffle)
        if train_p[TARGET_COL].nunique() < 2 or val_p[TARGET_COL].sum() == 0:
            continue
        sp      = StandardScaler()
        X_tr_p  = sp.fit_transform(train_p[FEATURE_COLS])
        X_val_p = sp.transform(val_p[FEATURE_COLS])
        clf_p   = clone(MODELS_REGISTRY["LogReg"])
        clf_p.fit(X_tr_p, train_p[TARGET_COL])
        proba_p = clf_p.predict_proba(X_val_p)[:, 1]
        if val_p[TARGET_COL].nunique() > 1:
            fold_aucs.append(roc_auc_score(val_p[TARGET_COL], proba_p))

    if fold_aucs:
        perm_mean_aucs.append(np.mean(fold_aucs))

perm_arr = np.array(perm_mean_aucs)
real_auc = summary.loc["LogReg", "auc_roc_mean"]
p_val    = (perm_arr >= real_auc).mean()

print(f"=== Permutation Test  (N={N_PERMUTATIONS}, model=LogReg) ===\n")
print(f"Real mean AUC-ROC:          {real_auc:.4f}")
print(f"Permuted mean AUC-ROC:      {perm_arr.mean():.4f} ± {perm_arr.std():.4f}")
print(f"95th pct of permutations:   {np.percentile(perm_arr, 95):.4f}")
print(f"p-value (one-sided):        {p_val:.4f}")
print(f"Significant at alpha=0.05:  {p_val < 0.05}")

=== Permutation Test  (N=100, model=LogReg) ===

Real mean AUC-ROC:          0.7525
Permuted mean AUC-ROC:      0.5017 ± 0.0582
95th pct of permutations:   0.5962
p-value (one-sided):        0.0000
Significant at alpha=0.05:  True


## 8. Merchant Health Scorecard

Final model trained on all months except the last, scored on the most recent month.

In [24]:
# Pick best model by AUC-PR (more informative than AUC-ROC for imbalanced targets)
best_model_name = summary["auc_pr_mean"].idxmax()
print(f"Best model by AUC-PR: {best_model_name}\n")

last_t  = len(usable_months) - 1
train_f = df_model[df_model["t_idx"] <  last_t]
score_f = df_model[df_model["t_idx"] == last_t].copy()

scaler_f  = StandardScaler()
X_train_f = scaler_f.fit_transform(train_f[FEATURE_COLS])
X_score_f = scaler_f.transform(score_f[FEATURE_COLS])

best_clf = clone(MODELS_REGISTRY[best_model_name])
if HAS_XGB and best_model_name == "XGBoost" and train_f[TARGET_COL].sum() > 0:
    best_clf.set_params(
        scale_pos_weight=(train_f[TARGET_COL] == 0).sum() / train_f[TARGET_COL].sum()
    )
best_clf.fit(X_train_f, train_f[TARGET_COL])

score_f["churn_proba"] = best_clf.predict_proba(X_score_f)[:, 1]
score_f["churn_flag"]  = (score_f["churn_proba"] >= 0.5).astype(int)

scorecard = score_f[[
    "merchant", "year_month",
    "total_amt_gross", "total_txs",
    "mom_volume_1m", "vol_vs_roll3",
    "churn_proba", "churn_flag", "churn",
]].sort_values("churn_proba", ascending=False).reset_index(drop=True)

print(f"=== Merchant Health Scorecard — {usable_months[-1]} ({best_model_name}) ===\n")
print(scorecard.to_string(index=False))

Best model by AUC-PR: GradBoost

=== Merchant Health Scorecard — 2026-02 (GradBoost) ===

              merchant year_month  total_amt_gross  total_txs  mom_volume_1m  vol_vs_roll3  churn_proba  churn_flag  churn
              Temu.com    2026-02  77,197,381.1000     159524        -0.6659        0.4542       0.9613           1      1
  Blizzard - Braintree    2026-02     355,671.3800        835        -0.6755        0.5061       0.6478           1      0
 GarenaDeltaForceLatam    2026-02       8,848.0000         45        -0.6212        0.4523       0.4552           0      0
          Alibaba ICBU    2026-02  87,410,692.2500       8483         0.9574        1.5388       0.0497           0      0
      Amazon Retail MX    2026-02 232,441,084.5200     205470        -0.2943        0.8531       0.0374           0      0
           Google Play    2026-02  96,671,608.2500    1010309         0.3024        1.6182       0.0327           0      0
TikTok Ads (SG) - 6241    2026-02   4,647,839.280

## 9. Feature Importance

In [25]:
print("=== Feature Importance — all models (trained on all-but-last-month) ===\n")

imp_data = {}
for name, base_clf in MODELS_REGISTRY.items():
    clf = clone(base_clf)
    if HAS_XGB and name == "XGBoost" and train_f[TARGET_COL].sum() > 0:
        clf.set_params(
            scale_pos_weight=(train_f[TARGET_COL] == 0).sum() / train_f[TARGET_COL].sum()
        )
    clf.fit(X_train_f, train_f[TARGET_COL])
    if hasattr(clf, "coef_"):
        imp_data[name] = clf.coef_[0]
    elif hasattr(clf, "feature_importances_"):
        imp_data[name] = clf.feature_importances_

imp_df = pd.DataFrame(imp_data, index=FEATURE_COLS)
imp_df["avg_abs"] = imp_df.abs().mean(axis=1)
imp_df = imp_df.sort_values("avg_abs", ascending=False).drop(columns="avg_abs")
print(imp_df.round(4).to_string())

=== Feature Importance — all models (trained on all-but-last-month) ===

                        LogReg  RandomForest  GradBoost  XGBoost  CatBoost
mom_txs_1m             -0.1497        0.1462     0.3415   0.1351   19.2090
mom_volume_1m          -0.1608        0.1449     0.0715   0.0851   17.9299
txs_vs_roll3           -0.3952        0.1457     0.0749   0.1084   11.1270
vol_vs_roll3           -0.2615        0.0819     0.0404   0.0645    8.1674
roll3_vol_cv            0.2588        0.0722     0.0431   0.0703    7.4062
lag1_total_amt_gross   -0.3017        0.0694     0.0306   0.0462    7.1343
lag2_total_amt_gross    0.1800        0.0529     0.0439   0.0719    6.6543
roll3_txs_cv            0.5413        0.0618     0.0120   0.0605    5.6530
lag1_refund_ratio       0.3590        0.0472     0.1519   0.0406    4.0399
roll3_txs_mean          0.0531        0.0319     0.0218   0.0630    4.2335
roll3_vol_mean         -0.0926        0.0521     0.0512   0.1184    3.6404
lag2_total_txs          0.2